In [1]:
from pathlib import Path
import os
import time
import random
import threading
import gc
import torch

import numpy as np
import pandas as pd

from memory_tracker import RSSPeakTracker
from training_utilities import profile_training
from training import train_3out, train_single
from data_helpers import make_loader_multi
from models_transformer import TwoHeadTransformerNet, SingleOutTransformerNet
from metrics import eval_split_metrics_3out_direct, eval_single_metrics

In [2]:
OUTDIR = Path("./Results_TR")
OUTDIR.mkdir(parents=True, exist_ok=True)
MODELSDIR = Path("./trained_models")
MODELSDIR.mkdir(parents=True, exist_ok=True)

In [3]:
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
print("Device:", DEVICE)

Device: cpu


#### Load data

In [4]:
train_data = np.load("data_processed/train_data.npz")
val_data = np.load("data_processed/val_data.npz")
test_data = np.load("data_processed/test_data.npz")

X_train = train_data["x"]
age_train = train_data["age"]
mets_train = train_data["mets"]
sex_train = train_data["sex"]

X_val = val_data["x"]
age_val = val_data["age"]
mets_val = val_data["mets"]
sex_val = val_data["sex"]

X_test = test_data["x"]
age_test = test_data["age"]
mets_test = test_data["mets"]
sex_test = test_data["sex"]

In [5]:
train_data_s = np.load("data_processed/train_data_scaled.npz")
val_data_s = np.load("data_processed/val_data_scaled.npz")
test_data_s = np.load("data_processed/test_data_scaled.npz")

X_train_s = train_data_s["x"]
age_train_s = train_data_s["age"]
mets_train_s = train_data_s["mets"]
sex_train_s = train_data_s["sex"]

X_val_s = val_data_s["x"]
age_val_s = val_data_s["age"]
mets_val_s = val_data_s["mets"]
sex_val_s = val_data_s["sex"]

X_test_s = test_data_s["x"]
age_test_s = test_data_s["age"]
mets_test_s = test_data_s["mets"]
sex_test_s = test_data_s["sex"]

In [6]:
scalers = np.load("data_processed/scalers.npz")

x_mean, x_std = scalers["x_mean"], scalers["x_std"]
age_mean, age_std = scalers["age_mean"], scalers["age_std"]
mets_mean, mets_std = scalers["mets_mean"], scalers["mets_std"]

##### Dataframes

In [7]:
BATCH_SIZE = 256

train_loader = make_loader_multi(X_train_s, age_train_s, mets_train_s, sex_train, BATCH_SIZE)
val_loader = make_loader_multi(X_val_s, age_val_s, mets_val_s, sex_val, BATCH_SIZE)
test_loader = make_loader_multi(X_test_s, age_test_s, mets_test_s, sex_test, BATCH_SIZE)

#### Models

In [8]:
IN_DIM = X_train_s.shape[1]

EMB_DIM = 64
NHEAD = 4
NUM_LAYERS = 2
FF_DIM = 128
DROPOUT = 0.1

tr_multioutput = TwoHeadTransformerNet(IN_DIM, emb_dim=EMB_DIM, nhead=NHEAD, 
                                       num_layers=NUM_LAYERS, ff_dim=FF_DIM, 
                                       dropout=DROPOUT).to(DEVICE)

### Training
#### Multioutput

In [9]:
EPOCHS = 1
METRICS_EVERY = int(EPOCHS/10)
LR = 1e-3
WD = 1e-6

# weights
LAMBDA_AGE = 1.0
LAMBDA_METS = 0.5
LAMBDA_SEX = 0.1

# M3 weights
LAMBDA_AGE_CONT = 1.0
LAMBDA_AGE_BIN = 0.8
LAMBDA_METS = 0.5
LAMBDA_SEX = 0.1

profile_rows = []

def _train_m3_call():
    return train_3out(
        tr_multioutput, train_loader, val_loader,         
        tag="TR_multioutput",
        device=DEVICE, savedir=OUTDIR,
        lr=LR, weight_decay=WD, epochs=EPOCHS,
        lambda_age=LAMBDA_AGE, lambda_mets=LAMBDA_METS, lambda_sex=LAMBDA_SEX,
        metrics_every=METRICS_EVERY, sex_threshold=0.5
    )


hist_m3, t_m3, pk_m3, st_m3 = profile_training(_train_m3_call, poll_every_sec=0.05)
profile_rows.append({
    "model_tag": "TR_multioutput",
    "train_seconds": t_m3,
    "start_rss_mb": st_m3,
    "peak_rss_mb": pk_m3,
    "rss_increase_mb": (pk_m3 - st_m3) if (not np.isnan(pk_m3) and not np.isnan(st_m3)) else np.nan
})

torch.save(tr_multioutput.state_dict(), MODELSDIR / "TR_multioutput.pt")

[TR_multioutput] ep=001 best_val=inf lr=1.00e-03 | loss va=0.3304 (age=0.2243, mets=0.1166, sex=0.4783)
   TRAIN: age(R2=0.537, RMSE=0.681, MAE=0.534) | MetSCORE(R2=0.754, RMSE=0.496, MAE=0.379) | sex(ACC=0.825, AUC=0.902, F1=0.854)
   VAL  : age(R2=0.501, RMSE=0.691, MAE=0.545) | MetSCORE(R2=0.758, RMSE=0.491, MAE=0.380) | sex(ACC=0.822, AUC=0.889, F1=0.853)
[TR_multioutput] ep=003 best_val=0.263776 lr=1.00e-03 | loss va=0.2543 (age=0.1911, mets=0.0889, sex=0.1867)
   TRAIN: age(R2=0.620, RMSE=0.617, MAE=0.482) | MetSCORE(R2=0.810, RMSE=0.436, MAE=0.306) | sex(ACC=0.939, AUC=0.983, F1=0.947)
   VAL  : age(R2=0.580, RMSE=0.634, MAE=0.497) | MetSCORE(R2=0.813, RMSE=0.432, MAE=0.300) | sex(ACC=0.929, AUC=0.979, F1=0.939)
[TR_multioutput] ep=006 best_val=0.233563 lr=1.00e-03 | loss va=0.2387 (age=0.1728, mets=0.0994, sex=0.1617)
   TRAIN: age(R2=0.667, RMSE=0.577, MAE=0.449) | MetSCORE(R2=0.791, RMSE=0.458, MAE=0.337) | sex(ACC=0.947, AUC=0.986, F1=0.955)
   VAL  : age(R2=0.621, RMSE=0.60

In [10]:
tr_multi_test = eval_split_metrics_3out_direct(tr_multioutput, 
                                               X_test_s, age_test_s, mets_test_s, 
                                               sex_test, DEVICE, sex_threshold=0.5)

multi_test_rows_all = []
multi_test_rows_all += [
    ["TR_multi", "age", tr_multi_test["age_R2"], tr_multi_test["age_RMSE"], 
     tr_multi_test["age_MAE"], np.nan, np.nan, np.nan],
    ["TR_multi", "MetSCORE", tr_multi_test["MetSCORE_R2"], 
     tr_multi_test["MetSCORE_RMSE"], tr_multi_test["MetSCORE_MAE"], np.nan, np.nan, np.nan],
    ["TR_multi", "sex", np.nan, np.nan, np.nan, tr_multi_test["sex_ACC"], 
     tr_multi_test["sex_AUC"], tr_multi_test["sex_F1"]],
]

metrics_tr_test_df = pd.DataFrame(multi_test_rows_all, 
                                  columns=["model_type", "output", 
                                           "R2", "RMSE", "MAE", "ACC", "AUC", "F1"]
)
metrics_save_path = OUTDIR / "test_metrics_TR_multioutput.csv"
metrics_tr_test_df.to_csv(metrics_save_path, index=False, float_format="%.6f")

In [11]:
metrics_tr_test_df

,model_type,output,R2,RMSE,MAE,ACC,AUC,F1
0,TR_multi,age,0.681722,0.569643,0.441764,NaN,NaN,NaN
1,TR_multi,MetSCORE,0.827210,0.410259,0.282814,NaN,NaN,NaN
2,TR_multi,sex,NaN,NaN,NaN,0.954831,0.987305,0.961783


#### Single output

In [12]:
single_specs = [
    ("age", "single_TR_age"),
    ("mets", "single_TR_MetSCORE"),
    ("sex", "single_TR_sex"),
]

single_test_rows = []

for kind, tag in single_specs:
    torch.manual_seed(SEED)    
    model_single = SingleOutTransformerNet(IN_DIM, emb_dim=EMB_DIM, nhead=NHEAD, 
                                           num_layers=NUM_LAYERS, ff_dim=FF_DIM, 
                                           dropout=DROPOUT).to(DEVICE)    
    
    def _train_single_call():
        return train_single(
            model_single, kind,
            train_loader, val_loader,
            tag=tag, device=DEVICE, savedir=OUTDIR,
            lr=LR, weight_decay=WD, epochs=EPOCHS,
            metrics_every=METRICS_EVERY,
            sex_threshold=0.5
        )

    hist_single, tr_time_s, pk_rss_mb, st_rss_mb = profile_training(_train_single_call, 
                                                                    poll_every_sec=0.05)
    profile_rows.append({
        "model_tag": f"TR-single-output:{kind}",
        "train_seconds": tr_time_s,
        "start_rss_mb": st_rss_mb,
        "peak_rss_mb": pk_rss_mb,
        "rss_increase_mb": (pk_rss_mb - st_rss_mb) if (not np.isnan(pk_rss_mb) and not np.isnan(st_rss_mb)) else np.nan
    })
    

    torch.save(model_single.state_dict(), MODELSDIR / f"TR_singleoutput_{kind}.pt")

    m_te = eval_single_metrics(model_single, kind, X_test_s, age_test_s, mets_test_s, sex_test, 
                               DEVICE, sex_threshold=0.5)

    if kind in ("age", "mets"):
        single_test_rows.append(["single-output(TR)", ("MetSCORE" if kind == "mets" else kind),
                                 m_te["R2"], m_te["RMSE"], m_te["MAE"], np.nan, np.nan, np.nan])
    else:
        single_test_rows.append(["single-output(TR)", kind,
                                 np.nan, np.nan, np.nan, m_te["ACC"], m_te["AUC"], m_te["F1"]])

[single_TR_age] ep=001 va_loss=0.2164 | TRAIN R2=0.546 | VAL R2=0.521
[single_TR_age] ep=003 va_loss=0.1839 | TRAIN R2=0.643 | VAL R2=0.593
[single_TR_age] ep=006 va_loss=0.1730 | TRAIN R2=0.668 | VAL R2=0.618
[single_TR_age] ep=009 va_loss=0.1714 | TRAIN R2=0.681 | VAL R2=0.625
[single_TR_age] ep=012 va_loss=0.1599 | TRAIN R2=0.702 | VAL R2=0.652
[single_TR_age] ep=015 va_loss=0.1652 | TRAIN R2=0.716 | VAL R2=0.639
[single_TR_age] ep=018 va_loss=0.1588 | TRAIN R2=0.729 | VAL R2=0.655
[single_TR_age] ep=021 va_loss=0.1703 | TRAIN R2=0.740 | VAL R2=0.625
[single_TR_age] ep=024 va_loss=0.1615 | TRAIN R2=0.752 | VAL R2=0.644
[single_TR_age] ep=027 va_loss=0.1674 | TRAIN R2=0.751 | VAL R2=0.631
[single_TR_age] ep=030 va_loss=0.1642 | TRAIN R2=0.766 | VAL R2=0.638
[single_TR_age] best val loss: 0.158644
[single_TR_MetSCORE] ep=001 va_loss=0.1259 | TRAIN R2=0.725 | VAL R2=0.729
[single_TR_MetSCORE] ep=003 va_loss=0.0824 | TRAIN R2=0.829 | VAL R2=0.829
[single_TR_MetSCORE] ep=006 va_loss=0.08

In [13]:
single_metrics_df = pd.DataFrame(
    single_test_rows,
    columns=["model_type", "output", "R2", "RMSE", "MAE", "ACC", "AUC", "F1"]
)
single_metrics_path = OUTDIR / "test_metrics_TR_singleoutput.csv"
single_metrics_df.to_csv(single_metrics_path, index=False, float_format="%.6f")

In [14]:
single_metrics_df

,model_type,output,R2,RMSE,MAE,ACC,AUC,F1
0,single-output(TR),age,0.698881,0.554075,0.431058,NaN,NaN,NaN
1,single-output(TR),MetSCORE,0.838405,0.396747,0.262103,NaN,NaN,NaN
2,single-output(TR),sex,NaN,NaN,NaN,0.944166,0.986201,0.952985


In [15]:
profile_df = pd.DataFrame(profile_rows)
profile_path = OUTDIR / "training_time_memory_profile.csv"
profile_df.to_csv(profile_path, index=False, float_format="%.6f")

In [16]:
profile_df

,model_tag,train_seconds,start_rss_mb,peak_rss_mb,rss_increase_mb
0,TR_multioutput,529.204761,541.687500,3097.476562,2555.789062
1,TR-single-output:age,526.790804,822.121094,3228.343750,2406.222656
2,TR-single-output:mets,521.879443,823.082031,3259.566406,2436.484375
3,TR-single-output:sex,537.197333,823.804688,3164.769531,2340.964844


In [17]:
def pick(tag):
    row = profile_df.loc[profile_df["model_tag"] == tag].copy()
    return None if row.empty else row.iloc[0]


mlp_multi_prof = pick("TR_multioutput")
s_age = pick("TR-single-output:age")
s_mets = pick("TR-single-output:mets")
s_sex = pick("TR-single-output:sex")

if (mlp_multi_prof is not None) and (s_age is not None) and (s_mets is not None) and (s_sex is not None):
    sum_single_time = float(s_age["train_seconds"] + s_mets["train_seconds"] + s_sex["train_seconds"])
    max_single_peak = float(np.nanmax([s_age["peak_rss_mb"], s_mets["peak_rss_mb"], s_sex["peak_rss_mb"]]))

    compare_df = pd.DataFrame([{
        "metric": "train_seconds",
        "TR_multi": float(mlp_multi_prof["train_seconds"]),
        "sum_single_TR": sum_single_time
    }, {
        "metric": "peak_rss_mb",
        "TR_multi": float(mlp_multi_prof["peak_rss_mb"]),
        "max_single_TR": max_single_peak
    }])

    compare_path = OUTDIR / "compare_TR_vs_sumSinglesTR_time_memory.csv"
    compare_df.to_csv(compare_path, index=False, float_format="%.6f")

In [18]:
compare_df

,metric,TR_multi,sum_single_TR,max_single_TR
0,train_seconds,529.204761,1585.86758,NaN
1,peak_rss_mb,3097.476562,NaN,3259.566406
